In [ ]:
# Darstellung anpassen

from IPython.display import display, HTML

display(HTML(data="""
<style>
    div#notebook-container    { width: 95%; }
    div#menubar-container     { width: 65%; }
    div#maintoolbar-container { width: 99%; }
</style>
"""))



In [ ]:
# Lade benötigte Programmpakete

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
import sklearn as sk
import joblib


In [ ]:
from sklearn.datasets import fetch_openml # Lade den MNIST_784-Datensatz
mnist=fetch_openml('mnist_784',as_frame=False,parser='auto')

In [ ]:
X=mnist['data'] # Features, 
y=mnist['target'] # Labels
print(f"Shape of X: {X.shape}")
# Teile in Trainings und Test-Daten. Verwende die ersten 60000 Bilder zum trainieren und die letzten 10000 zum Testen
X_train=X[:60000,:]
y_train=y[:60000]
X_test=X[60000:,:]
y_test=y[60000:]



In [ ]:
# Plotte Beispielbilder der Ziffern
fig,ax=plt.subplots(5,5,figsize=(10,10))
for i in range(25):
    j=i%5
    k=i//5

    test_image=X[i].reshape(28,28)
    ax[j,k].imshow(test_image,cmap='gray',vmin=0, vmax=255)

##### Entwickeln Sie einen Classifier der die Ziffer '5' erkennt. Verwenden Sie zur Entwicklung den Trainingsdatensatz X_train, y_train und verwenden Sie die in der LV besprochenen Methoden um ein optimales Modell mit optimalen Hyperparametern zu entwickeln. Erst wenn Sie fertig sind, und überzeugt das bestmögliche Modell gefunden zu haben, Werten Sie ihre Daten am Test-Datensatz aus. 

In [ ]:
# Preprocsessing
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
print(scaler.fit(X_train))
print(scaler.data_max_)
print(scaler.transform(X_train))
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 
y_train_binary = (y_train == "5") # why is this a string???????
y_test_binary = (y_test == "5")


In [ ]:
# basic training runs in single core => can take a while; C=10 ~ 2 minutes, used for prep
from sklearn.svm import SVC
basic_svc=SVC(C=10) # probability=True
basic_svc.fit(X_train_scaled, y_train_binary)
joblib.dump(basic_svc, "models/svc_model_binary_basic.joblib")

Finding optimal Hyperparameters by using GridSearchCV, even when using a low amout of splits, parameters training takes a lot of time, even with parallization...      
1fit with C=10 ~ 1:40


In [ ]:
# binary => 1 Stunde Rechenzeiot Octa Core
from sklearn.model_selection import GridSearchCV, KFold

svc=SVC(kernel='rbf') 
cs = np.logspace(1, 3, 4) 
# gammas = [1/600000, 1/60000, 1/6000] # 1/n_samples default value
params = {
    'C': cs,
    # 'gamma': gammas,
}

gs_binary = GridSearchCV(svc, params, cv=KFold(n_splits=4), scoring = "balanced_accuracy", verbose=3, n_jobs=-1)#n_jobs=-1 => alle Kerne nutzen
gs_binary.fit(X_train_scaled, y_train_binary)

joblib.dump(gs_binary, "models/svc_model_binary_good.joblib")

Aufgrund der des großen Trainingsdatensatzes kommt es bie hohen C/gamma zu extrem hohen Rechenzeiten.

In [ ]:
# Optimale Parameter
print(f"Best score: {gs_binary.best_params_}")

In [ ]:
# non biary, all digits
from sklearn.model_selection import GridSearchCV, KFold

svc=SVC(kernel='rbf')# probability=True

cs = np.logspace(-1, 3, 4) 
# gammas = [1/600000, 1/60000, 1/6000] # 1/n_samples default value
params = {
    'C': cs,
    # 'gamma': gammas,
}

gs = GridSearchCV(svc, params, cv=KFold(n_splits=4), scoring = "balanced_accuracy", verbose=3, n_jobs=-1)#n_jobs=-1 => alle Kerne nutzen, einen thread für
gs.fit(X_train_scaled, y_train)

joblib.dump(gs, "models/svc_model_good.joblib")

In [ ]:
svc = joblib.load("models/svc_model_binary_good.joblib")
# Optimale Parameter
print(f"Best score: {svc.best_params_}")

Um die Rechenzeit für die Auswertung des Modells zu reduzieren werden Test und Trainingsdatensatz auf alle Kerne der CPU aufgeteilt um eine parallele berechnung zu ermöglichen.

In [ ]:
import os
number_of_cpus = os.cpu_count()

In [ ]:
# 10 seconds
from joblib import Parallel, delayed

def batch_predict(model, X_batch):
    return model.predict(X_batch)

# Split data into 4 chunks
X_chunks = np.array_split(X_test_scaled, number_of_cpus)

# Predict in parallel
predictions = Parallel(n_jobs=number_of_cpus)(delayed(batch_predict)(svc, chunk) for chunk in X_chunks)

# Concatenate results
y_pred = np.concatenate(predictions)
accuracy = (y_pred == y_test_binary).mean()
print("Accuracy on test set:", accuracy)

In [ ]:
# 10 seconds 
X_chunks = np.array_split(X_train_scaled, number_of_cpus)

# Predict in parallel
predictions = Parallel(n_jobs=number_of_cpus-2)(delayed(batch_predict)(svc, chunk) for chunk in X_chunks)

# Concatenate results
y_pred_train = np.concatenate(predictions)
accuracy = (y_pred_train == y_train_binary).mean()
print("Accuracy on train set:", accuracy)

In [ ]:
#gpt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
confusion_matrix = confusion_matrix(y_test_binary, y_pred)
cmd = ConfusionMatrixDisplay(confusion_matrix=confusion_matrix, display_labels=svc.classes_)
cmd.plot(cmap=plt.cm.Blues, xticks_rotation='vertical')


In [ ]:
# Primt classifictaion report
from sklearn.metrics import classification_report

cr = classification_report(y_test, y_pred)
print(cr)

Fasle positives, False Negatives:

In [ ]:
false_positives = np.where((y_test_binary == 0) & (y_pred == 1))[0]
false_negatives = np.where((y_test_binary == 1) & (y_pred == 0))[0]
print("False Positives:", false_positives)
print("False Negatives:", false_negatives)

false_positives = false_positives[:5]  # Limit to 5 for display
false_negatives = false_negatives[:5]  # Limit to 5 for display

fig,ax=plt.subplots(1,5,figsize=(10,2))
fig.suptitle("False Positives")
for id, i in enumerate(false_positives):
    print(id, i)
    test_image=X_test[i].reshape(28,28)
    ax[id].imshow(test_image,cmap='gray',vmin=0, vmax=255)

fig,ax=plt.subplots(1,5,figsize=(10,2))
fig.suptitle("False Negatives")
for id, i in enumerate(false_negatives):
    test_image=X_test[i].reshape(28,28)
    ax[id].imshow(test_image,cmap='gray',vmin=0, vmax=255)

Einige der Fehler wären auch für einen Menschen plausibel. z.B.: 1. False Positive 

Alle Ziffern

In [ ]:
svc = joblib.load("models/svc_model_good.joblib")
# Optimale Parameter
print(f"Best score: {svc.best_params_}")

In [ ]:
from joblib import Parallel, delayed

def batch_predict(model, X_batch):
    return model.predict(X_batch)

X_chunks = np.array_split(X_test_scaled, number_of_cpus)

# Predict in parallel
predictions = Parallel(n_jobs=number_of_cpus)(delayed(batch_predict)(svc, chunk) for chunk in X_chunks)

# Concatenate results
y_pred = np.concatenate(predictions)
accuracy = (y_pred == y_test).mean()
print("Accuracy on test set:", accuracy)

In [ ]:
X_chunks = np.array_split(X_train_scaled, number_of_cpus-2)

# Predict in parallel
predictions = Parallel(n_jobs=number_of_cpus-2)(delayed(batch_predict)(svc, chunk) for chunk in X_chunks)

# Concatenate results
y_pred_train = np.concatenate(predictions)
accuracy = (y_pred_train == y_train).mean()
print("Accuracy on train set:", accuracy)

In [ ]:
y_pred

In [ ]:
y_pred

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import matplotlib.pyplot as plt

# Compute the confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Display it
cmd = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=svc.classes_)
cmd.plot(cmap=plt.cm.Blues, xticks_rotation='vertical')
plt.show()


In [ ]:
cr = classification_report(y_test, y_pred)
print(cr)